# Part 1.10 Datasets

After this notebook you can build a LangSmith dataset whose examples carry an image
attachment, inputs, and reference outputs.

In [ ]:
from kea.config import setup

setup("kea-claims-local")

## See it

How do we know a change made the workflow better? Run it on a fixed set of claims. The
`kea-claims` dataset has 14 bag photos with ground truth: damage type, severity, and
whether it is luggage at all. The photo is an attachment on each example.

<img src="../../assets/screenshots/p1-10-dataset.png" alt="the kea-claims dataset in LangSmith: one example open with its photo attachment, inputs, and reference outputs, and the split column visible." width="900">

## Build it

Start from the ground truth: one row per photo, with its split.

In [ ]:
from kea.evals.claims import load_ground_truth

for truth in load_ground_truth():
    print(
        f"{truth['id']:<13} {truth['split']:<10} {truth['damage_type']:<14} "
        f"severity {truth['severity']}  luggage={truth['is_luggage']}"
    )

Create the dataset if it doesn't exist, then add one example per photo that isn't in it
yet. The photo goes in as an **attachment**; the form fields are the inputs; the
truth is the reference output.

In [ ]:
from langsmith import Client
from langsmith.schemas import ExampleCreate

from kea.claims.images import IMAGE_ATTACHMENT, PHOTO_DIR, mime_for
from kea.evals.claims import DATASET_NAME, claim_form

client = Client()
if not client.has_dataset(dataset_name=DATASET_NAME):
    client.create_dataset(
        dataset_name=DATASET_NAME,
        description="Kea Air baggage-claim photos with a claim form and ground-truth damage.",
    )
dataset = client.read_dataset(dataset_name=DATASET_NAME)
have = {(e.metadata or {}).get("photo_id") for e in client.list_examples(dataset_id=dataset.id)}

In [ ]:
new = []
for index, truth in enumerate(load_ground_truth()):
    if truth["id"] in have:
        continue
    photo = PHOTO_DIR / truth["file"]
    new.append(
        ExampleCreate(
            inputs=claim_form(index, truth["id"]),
            outputs={k: truth[k] for k in ("is_luggage", "damage_type", "severity")},
            metadata={"photo_id": truth["id"], "note": truth["note"]},
            split=truth["split"],
            attachments={IMAGE_ATTACHMENT: (mime_for(photo), photo.read_bytes())},
        )
    )
if new:
    client.create_examples(dataset_id=dataset.id, examples=new)

print(f"added {len(new)} examples")
print(dataset.url)

In [ ]:
example = next(client.list_examples(dataset_id=dataset.id, limit=1, include_attachments=True))
print("inputs: ", example.inputs)
print("outputs:", example.outputs)
print("split:  ", example.metadata and example.metadata.get("dataset_split"))
print("attachments:", list(example.attachments or {}))

## Under the hood

- Splits come from the ground truth: `happy` (clear cases), `ambiguous` (blurry,
  scuffed), and `guardrail` (not luggage). Run one split at a time to see where the
  workflow breaks.
- Inputs match the graph's input schema (`ClaimInput`), so an example can be sent
  straight to the graph. The photo comes in separately as an attachment.
- `claim_form` makes the form fields deterministically, so reruns build the same
  dataset.
- `kea.evals.claims.build_dataset` does this and also updates existing examples in
  place, so it is safe to run again.
- Datasets can also be built from production traces. The photo on the root run
  (Part 1.5) is why a trace can be added as an example.

Docs: [Manage datasets](https://docs.langchain.com/langsmith/manage-datasets).

## Recap

- A dataset is inputs plus reference outputs, here with the photo as an attachment.
- Splits let you score easy, ambiguous, and guardrail cases separately.
- Build it from code so it is repeatable.

**Next:** [Part 1.11 Evaluators](11_evaluators.ipynb)